# Units TopK SAE retraining and force-specific confirmation

This is the final guarded units experiment. It applies the same fixed TopK `k=128,256,512` sweep and metric-only selection used for mathematics, then performs final-token causal validation. The primary confirmation compares force-sourced feature swaps against matched mass-sourced swaps into the same unseen energy targets.

Only the predeclared Top-10 panel is confirmatory. Other panel sizes, layers and controls are secondary diagnostics and must not replace the primary result after outcomes are viewed.

## 1. Mount Drive and fetch the current repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

repo_url = 'https://github.com/evey-dev/test_run.git'
repo_dir = '/content/test_run'

def run_cmd(command):
    print('$', ' '.join(map(str, command)))
    subprocess.run(command, check=True)

github_ok = False
try:
    checkout = repo_dir
    if os.path.isdir(os.path.join(checkout, '.git')):
        run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
    else:
        if os.path.exists(checkout) and os.listdir(checkout):
            checkout = '/content/test_run_github'
        if os.path.isdir(os.path.join(checkout, '.git')):
            run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
        elif os.path.exists(checkout) and os.listdir(checkout):
            raise RuntimeError(f'{checkout} exists but is not a git repository')
        else:
            run_cmd(['git', 'clone', '--depth', '1', repo_url, checkout])
    os.chdir(checkout)
    github_ok = True
    print('Using GitHub checkout:', os.getcwd())
except Exception as exc:
    print('GitHub checkout failed; using Drive project.zip backup.')
    print(repr(exc))

if not github_ok:
    zip_path = '/content/drive/MyDrive/mphil-project/project.zip'
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f'Could not find {zip_path}')
    run_cmd(['unzip', '-q', '-o', zip_path, '-d', '/content/'])
    for candidate in ['/content/test_run', '/content/mphil_project/test_run', '/content']:
        if os.path.isdir(os.path.join(candidate, 'src')) and os.path.isdir(os.path.join(candidate, 'configs')):
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError('Could not locate the extracted repository root')

print('Current working directory:', os.getcwd())

## 2. Install dependencies and regenerate deterministic data

In [ ]:
!pip install -q --upgrade "transformers>=4.51.0" accelerate matplotlib pandas
!pip install -q -e .
!python data/generate_datasets.py --capitals

from pathlib import Path
required_files = [
    Path('src/units_feature_screen.py'),
    Path('src/plot_topk_units.py'),
    Path('configs/sae_units_topk128_config.yaml'),
    Path('configs/sae_units_topk256_config.yaml'),
    Path('configs/sae_units_topk512_config.yaml'),
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        'The GitHub checkout is missing the units TopK follow-up files. Push first: ' + ', '.join(missing)
    )

## 3. Fixed candidates and artifact paths

In [ ]:
from pathlib import Path
import json
import shutil
import sys

LAYERS = [4, 8, 12, 16, 20, 24, 28]
CANDIDATES = {
    128: 'configs/sae_units_topk128_config.yaml',
    256: 'configs/sae_units_topk256_config.yaml',
    512: 'configs/sae_units_topk512_config.yaml',
}
DRIVE_ROOT = Path('/content/drive/MyDrive/mphil-project')
DRIVE_CHECKPOINT_ROOT = DRIVE_ROOT / 'mechanistic_data' / 'topk_units_retrain'
DRIVE_OUTPUT = DRIVE_ROOT / 'outputs' / 'topk_units_retrain'
LOCAL_OUTPUT = Path('outputs/topk_units_retrain')
FIGURE_OUTPUT = LOCAL_OUTPUT / 'figures'
for path in (DRIVE_CHECKPOINT_ROOT, DRIVE_OUTPUT, LOCAL_OUTPUT, FIGURE_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)

print('Candidates:', CANDIDATES)
print('Drive checkpoint root:', DRIVE_CHECKPOINT_ROOT)
print('Drive output root:', DRIVE_OUTPUT)

## 4. Recapture the units final-token activation corpus

This deliberately matches the original units SAE training support. The experiment changes sparsity and decoder normalisation, not the prompt corpus or activation position.

In [ ]:
activation_dir = Path('mechanistic_data_units_topk_retrain')
activation_complete = (activation_dir / 'train_val_indices_per_layer.npy').exists() and all(
    (activation_dir / f'activations_layer{layer}.npy').exists() for layer in LAYERS
)
if activation_complete:
    print('Units TopK activation corpus already exists:', activation_dir)
else:
    run_cmd([
        sys.executable, '-m', 'src.capture_activations',
        '--output-dir', str(activation_dir),
        '--behaviours', 'units',
        '--layers', *map(str, LAYERS),
        '--seed', '787',
    ])

## 5. Train or restore all fixed TopK candidates

In [ ]:
for top_k, config in CANDIDATES.items():
    drive_dir = DRIVE_CHECKPOINT_ROOT / f'k{top_k}'
    drive_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n=== Training/restoring units TopK {top_k} ===')
    run_cmd([
        sys.executable, '-m', 'src.train',
        '--config', config,
        '--drive-dir', str(drive_dir),
    ])

## 6. Select one candidate before intervention

Selection requires mean validation FVE at least 0.90, every layer FVE at least 0.85, and mean dead-feature fraction at most 0.80. The eligible candidate with the smallest mean validation L0 is selected without using any graph or intervention outcome.

In [ ]:
diagnostic_csvs = []
for top_k, config in CANDIDATES.items():
    csv_path = LOCAL_OUTPUT / f'units_topk{top_k}_diagnostics.csv'
    json_path = LOCAL_OUTPUT / f'units_topk{top_k}_diagnostics.json'
    run_cmd([
        sys.executable, '-m', 'src.sae_diagnostics',
        '--config', config,
        '--label', f'units_topk{top_k}',
        '--output-json', str(json_path),
        '--output-csv', str(csv_path),
        '--device', 'auto',
    ])
    diagnostic_csvs.append(csv_path)

selection_path = LOCAL_OUTPUT / 'units_topk_selection.json'
run_cmd([
    sys.executable, '-m', 'src.select_sae_candidate',
    '--configs', *CANDIDATES.values(),
    '--diagnostics', *map(str, diagnostic_csvs),
    '--minimum-mean-fve', '0.90',
    '--minimum-layer-fve', '0.85',
    '--maximum-mean-dead-fraction', '0.80',
    '--output', str(selection_path),
])
selection = json.loads(selection_path.read_text())
if selection.get('selected') is None:
    for source in LOCAL_OUTPUT.glob('units_topk*_diagnostics.*'):
        shutil.copy2(source, DRIVE_OUTPUT / source.name)
    shutil.copy2(selection_path, DRIVE_OUTPUT / selection_path.name)
    raise RuntimeError('No units TopK candidate passed the predeclared thresholds; stop before graph construction.')
SELECTED_K = int(selection['selected']['top_k'])
SELECTED_CONFIG = selection['selected']['config']
print('Selected units TopK candidate:', SELECTED_K, SELECTED_CONFIG)

import pandas as pd
display(pd.DataFrame([
    {
        'k': row['top_k'],
        'mean validation FVE': row['mean_validation_fve'],
        'minimum layer FVE': row['minimum_layer_validation_fve'],
        'mean validation L0': row['mean_validation_l0'],
        'dead-feature fraction': row['mean_dead_feature_fraction'],
        'selected': row['config'] == selection['selected']['config'],
    }
    for row in selection['candidates']
]).round(4))

## 7. Restore optional ReLU references and build exactly one selected graph

In [ ]:
for name in ('final_sae_diagnostics_units.csv', 'units_final_heldout_last.json'):
    local = Path('outputs') / name
    drive_source = DRIVE_ROOT / 'outputs' / name
    if not local.exists() and drive_source.exists():
        local.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_source, local)
        print('Restored reference:', name)

GRAPH_STEM = f'units_topk{SELECTED_K}_force_graph'
graph_paths = {suffix: LOCAL_OUTPUT / f'{GRAPH_STEM}.{suffix}' for suffix in ('json', 'html', 'md')}
drive_graph_paths = {suffix: DRIVE_OUTPUT / f'{GRAPH_STEM}.{suffix}' for suffix in graph_paths}
if all(path.exists() for path in drive_graph_paths.values()):
    for suffix in graph_paths:
        shutil.copy2(drive_graph_paths[suffix], graph_paths[suffix])
    print('Restored selected units graph from Drive:', GRAPH_STEM)
else:
    run_cmd([
        sys.executable, '-m', 'src.attribution_graph',
        '--prompt', 'Fact: The official SI unit for the force of a moving engine thrust is named "',
        '--target', 'newtons',
        '--contrast-target', 'joules',
        '--layers', *map(str, LAYERS),
        '--sae-config', SELECTED_CONFIG,
        '--top-k-nodes', '20',
        '--top-k-edges', '30',
        '--output-json', str(graph_paths['json']),
        '--output-html', str(graph_paths['html']),
        '--output-mermaid', str(graph_paths['md']),
    ])
    for suffix in graph_paths:
        shutil.copy2(graph_paths[suffix], drive_graph_paths[suffix])

## 8. Final-token graph-held-out benchmark on corpus validation contexts

In [ ]:
heldout_path = LOCAL_OUTPUT / f'units_topk{SELECTED_K}_heldout_last.json'
run_cmd([
    sys.executable, '-m', 'src.heldout_validation',
    '--units-sae-config', SELECTED_CONFIG,
    '--units-graph', str(graph_paths['json']),
    '--unit-cases', '20',
    '--skip-math',
    '--positions', 'last',
    '--output', str(heldout_path),
])
heldout = json.loads(heldout_path.read_text())
display(pd.DataFrame(heldout['units']['summary']['conditions']).T.round(4))

relu_heldout_path = Path('outputs/units_final_heldout_last.json')
if relu_heldout_path.exists():
    relu_heldout = json.loads(relu_heldout_path.read_text())
    comparison = pd.DataFrame({
        'Original ReLU': {
            name: values['mean_gap_delta']
            for name, values in relu_heldout['units']['summary']['conditions'].items()
        },
        f'TopK {SELECTED_K}': {
            name: values['mean_gap_delta']
            for name, values in heldout['units']['summary']['conditions'].items()
        },
    })
    display(comparison.round(4))

## 9. Fresh-context discovery and confirmation screen

All force, mass and energy prompts in this screen are exact-string absent from the SAE corpus. Discovery uses eight physical systems and confirmation uses sixteen different systems, with one prompt per system. The output is checkpointed directly to Drive after every feature and panel; rerun this cell after interruption to resume.

In [ ]:
drive_screen_path = DRIVE_OUTPUT / f'units_topk{SELECTED_K}_feature_screen.json'
run_cmd([
    sys.executable, '-m', 'src.units_feature_screen',
    '--sae-config', SELECTED_CONFIG,
    '--graph', str(graph_paths['json']),
    '--positions', 'last',
    '--discovery-cases', '8',
    '--confirmation-cases', '16',
    '--seed', '2787',
    '--panel-sizes', '1', '3', '5', '10', '20',
    '--primary-panel-size', '10',
    '--random-panels', '5',
    '--output', str(drive_screen_path),
])
local_screen_path = LOCAL_OUTPUT / drive_screen_path.name
shutil.copy2(drive_screen_path, local_screen_path)
screen = json.loads(local_screen_path.read_text())
primary = screen['confirmation']['primary_result']
print('Status:', screen['status'])
print('Candidate features:', screen['candidate_feature_count'])
print('Primary panel:', primary['panel'])
print('Predeclared success rule met:', primary['supports_force_selectivity_under_predeclared_rule'])
display(pd.Series(primary['summary']).to_frame('value'))
display(pd.DataFrame([
    {
        'panel': panel['name'],
        'kind': panel['kind'],
        'features': panel['feature_count'],
        'force delta': panel['summary']['mean_force_source_delta'],
        'mass delta': panel['summary']['mean_mass_source_delta'],
        'force minus mass': panel['summary']['mean_force_minus_mass_difference'],
        'paired 95% CI': panel['summary']['bootstrap_95_ci_mean_force_minus_mass_difference'],
    }
    for panel in screen['confirmation']['panels']
]))

## 10. Generate the report figure and persist all outputs

In [ ]:
plot_command = [
    sys.executable, '-m', 'src.plot_topk_units',
    '--candidate-diagnostics', *map(str, diagnostic_csvs),
    '--selection', str(selection_path),
    '--screen', str(local_screen_path),
    '--output-dir', str(FIGURE_OUTPUT),
]
original_diagnostics = Path('outputs/final_sae_diagnostics_units.csv')
if original_diagnostics.exists():
    plot_command.extend(['--original-diagnostics', str(original_diagnostics)])
run_cmd(plot_command)

from IPython.display import Image, display
display(Image(filename=str(FIGURE_OUTPUT / 'fig_units_topk_selection_confirmation.png')))

for source in LOCAL_OUTPUT.rglob('*'):
    if source.is_file():
        destination = DRIVE_OUTPUT / source.relative_to(LOCAL_OUTPUT)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        print('Copied', source.relative_to(LOCAL_OUTPUT))

print('Units TopK follow-up complete.')
print('Selected k:', SELECTED_K)
print('Drive outputs:', DRIVE_OUTPUT)

## Interpretation gate

- A positive primary result requires both the force-source effect and the force-minus-mass specificity effect to have bootstrap 95% intervals wholly above zero on the untouched confirmation contexts.
- If the primary rule succeeds and matched random panels do not reproduce it, report a compact, final-position force-selective feature panel.
- If the primary rule fails, do not select a secondary panel or another `k`. Report the position-consistent ReLU result and the failed TopK compactness test, then stop experimentation.